# Phần 5 — Mô hình phân loại dựa trên LSTM/GRU

**Ý tưởng sáng tạo:** Ảnh không phải chuỗi thời gian,
nhưng ta có thể "đọc" ảnh theo nhiều cách:
- Đọc từng hàng (như đọc sách)
- Đọc từng cột
- Đọc từng patch (như quét QR code)

Sau đó dùng **LSTM** hoặc **GRU** xử lý chuỗi này để phân loại.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from src.data import get_cifar100_loaders, get_sequence_loaders, get_device
from src.models_part5 import ImageLSTM, ImageGRU
from src.train import fit, load_best_model
from src.utils import (get_param_count, get_predictions, compute_metrics,
                       plot_multi_curves, plot_comparison_bar, print_results_table,
                       save_metrics_json, load_metrics_json)

DEVICE = get_device()
print(f"Thiết bị: {DEVICE}")

## 1. Các cách biểu diễn ảnh thành chuỗi

In [ ]:
# Visualize 4 cách đọc ảnh
_, _, test_loader_img, class_names = get_cifar100_loaders(batch_size=8)
images, labels = next(iter(test_loader_img))
img = images[0]  # [3, 32, 32]
CIFAR100_MEAN = torch.tensor([0.5071, 0.4867, 0.4408])
CIFAR100_STD  = torch.tensor([0.2675, 0.2565, 0.2761])
img_show = (img * CIFAR100_STD[:,None,None] + CIFAR100_MEAN[:,None,None]).clamp(0,1)
img_np = img_show.permute(1,2,0).numpy()

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

# Row-wise
axes[0].imshow(img_np)
for i in range(0, 32, 4):
    axes[0].axhline(y=i-0.5, color='yellow', linewidth=1, alpha=0.7)
arrow = mpatches.FancyArrowPatch((0, 0), (31, 0), arrowstyle='->', color='yellow', linewidth=2)
axes[0].add_patch(arrow)
axes[0].set_title(f'Row-wise\nT=32, D=96\n(32 hàng, mỗi hàng 3×32=96)')
axes[0].axis('off')

# Col-wise
axes[1].imshow(img_np)
for j in range(0, 32, 4):
    axes[1].axvline(x=j-0.5, color='cyan', linewidth=1, alpha=0.7)
axes[1].set_title(f'Col-wise\nT=32, D=96\n(32 cột, mỗi cột 3×32=96)')
axes[1].axis('off')

# Patch4
axes[2].imshow(img_np)
for i in range(8):
    for j in range(8):
        rect = mpatches.Rectangle((j*4-0.5, i*4-0.5), 4, 4,
                                    linewidth=1, edgecolor='red', facecolor='none')
        axes[2].add_patch(rect)
        if i == 0 and j < 3:
            axes[2].text(j*4+1.5, 1.5, f'{i*8+j+1}', fontsize=6, color='white', ha='center')
axes[2].set_title('Patch4 (4×4)\nT=64, D=48\n(64 patches, mỗi patch 3×4×4=48)')
axes[2].axis('off')

# Patch8
axes[3].imshow(img_np)
for i in range(4):
    for j in range(4):
        rect = mpatches.Rectangle((j*8-0.5, i*8-0.5), 8, 8,
                                    linewidth=2, edgecolor='green', facecolor='none')
        axes[3].add_patch(rect)
        axes[3].text(j*8+3.5, i*8+3.5, f'{i*4+j+1}', fontsize=9, color='white',
                    ha='center', va='center', fontweight='bold')
axes[3].set_title('Patch8 (8×8)\nT=16, D=192\n(16 patches, mỗi patch 3×8×8=192)')
axes[3].axis('off')

plt.suptitle('4 cách đọc ảnh CIFAR-100 thành chuỗi cho LSTM/GRU', fontsize=12)
plt.tight_layout()
plt.show()

## 2. So sánh LSTM và GRU

| | LSTM | GRU |
|--|------|-----|
| **Gates** | 4 (forget, input, output, cell) | 2 (reset, update) |
| **Cell state** | Có (c_t) — memory dài hạn | Không |
| **Parameters** | ~4×(D+H)×H mỗi layer | ~3×(D+H)×H mỗi layer |
| **Tốc độ** | Chậm hơn | Nhanh hơn ~25% |
| **Accuracy** | Thường tương đương | Thường tương đương |

Với D=96 (row), H=256, num_layers=2, bidirectional:

In [ ]:
# So sánh số params
for seq_mode, input_size in [("row", 96), ("patch4", 48)]:
    lstm = ImageLSTM(input_size=input_size)
    gru = ImageGRU(input_size=input_size)
    print(f"seq_mode={seq_mode} (input_size={input_size}):")
    print(f"  LSTM params: {get_param_count(lstm)}")
    print(f"  GRU  params: {get_param_count(gru)}")
    print()

## 3. Tải dữ liệu dạng chuỗi

In [ ]:
# Tải 4 loaders khác nhau
loaders = {}
seq_configs = [
    ("LSTM-row",   "lstm",  "row",    96),
    ("LSTM-patch4","lstm",  "patch4", 48),
    ("GRU-row",    "gru",   "row",    96),
    ("GRU-patch4", "gru",   "patch4", 48),
]

for name, rnn_type, seq_mode, input_size in seq_configs:
    print(f"\nLoading {name} ({seq_mode})...")
    tr, vl, te, seq_len, inp_size = get_sequence_loaders(seq_mode=seq_mode, batch_size=128)
    loaders[name] = {"train": tr, "val": vl, "test": te,
                     "seq_len": seq_len, "input_size": inp_size,
                     "rnn_type": rnn_type}

## 4. Huấn luyện

In [ ]:
TRAIN_MODE = True
histories_p5 = {}

for name, cfg in loaders.items():
    ckpt_path = f'../exercise/results/checkpoints/{name.lower()}.pt'
    history_path = f'../exercise/results/metrics/{name.lower()}_history.json'

    RNNCls = ImageLSTM if cfg["rnn_type"] == "lstm" else ImageGRU
    model = RNNCls(input_size=cfg["input_size"])

    if TRAIN_MODE:
        print(f"\n{'='*50}\nTraining: {name} (T={cfg['seq_len']}, D={cfg['input_size']})\n{'='*50}")
        model = model.to(DEVICE)
        hist = fit(model, cfg["train"], cfg["val"],
                   {"epochs": 30, "lr": 1e-3, "device": DEVICE, "save_path": ckpt_path})
        histories_p5[name] = hist
        save_metrics_json(hist, history_path)
    else:
        histories_p5[name] = load_metrics_json(history_path)

    print(f"✓ {name} done")

In [ ]:
# Training curves comparison
plot_multi_curves(
    list(histories_p5.values()),
    list(histories_p5.keys()),
    title="So sánh LSTM/GRU × Sequence Representation (Phần 5)",
    save_path='../exercise/results/plots/part5_comparison_curves.png'
)

In [ ]:
# Bảng kết quả
results_p5 = {}
for name, cfg in loaders.items():
    ckpt_path = f'../exercise/results/checkpoints/{name.lower()}.pt'
    if os.path.exists(ckpt_path):
        RNNCls = ImageLSTM if cfg["rnn_type"] == "lstm" else ImageGRU
        m = load_best_model(RNNCls(cfg["input_size"]), ckpt_path, DEVICE)
        preds, labels = get_predictions(m, cfg["test"], DEVICE)
        metrics = compute_metrics(preds, labels)
        results_p5[name] = {
            "test_acc": metrics["accuracy"],
            "val_acc": max(histories_p5[name]["val_acc"]),
            "f1_macro": metrics["f1_macro"],
            "params": get_param_count(m),
        }

save_metrics_json(results_p5, '../exercise/results/metrics/part5_results.json')
print_results_table(results_p5)

plot_comparison_bar(results_p5, metric="test_acc",
                    title="Test Accuracy — Phần 5 (LSTM/GRU)",
                    save_path='../exercise/results/plots/part5_bar.png')

## 5. Grand Summary — Tất cả mô hình

So sánh toàn bộ ~12 mô hình từ 5 phần của bài tập.

In [ ]:
# Gộp tất cả results
all_results = {}

def safe_load(path):
    return load_metrics_json(path) if os.path.exists(path) else {}

all_results.update(safe_load('../exercise/results/metrics/part1_2_results.json'))
all_results.update(safe_load('../exercise/results/metrics/part3_results.json'))
all_results.update(safe_load('../exercise/results/metrics/part4_results.json'))
all_results.update(safe_load('../exercise/results/metrics/part5_results.json'))

if all_results:
    print("\n=== BẢNG TỔNG KẾT TẤT CẢ MÔ HÌNH ===")
    print_results_table(all_results)
    plot_comparison_bar(
        all_results, metric="test_acc",
        title="So sánh Tất cả Mô hình — CIFAR-100",
        save_path='../exercise/results/plots/grand_summary_bar.png'
    )

## 6. Nhận xét Phần 5 & Tổng kết

**LSTM/GRU so với CNN và ViT:**
- LSTM/GRU thường kém CNN vì ảnh có **cấu trúc không gian 2D**,
  không phải chuỗi 1D
- Đọc theo hàng/cột làm mất quan hệ dọc/ngang
- Patches nhỏ giữ được nhiều thông tin cục bộ hơn

**LSTM vs GRU:**
- Thường đạt accuracy tương đương
- GRU đơn giản hơn (ít params ~25%) → train nhanh hơn
- Không có quy tắc tuyệt đối: tùy bài toán

**Row-wise vs Patch-wise:**
- Patch-wise thường tốt hơn vì giữ được thông tin không gian cục bộ
- Patch 4×4 (T=64) cân bằng giữa số bước và độ phân giải

**Kết luận chung (rank dự kiến):**
1. 🥇 SimpleCNN (~55%)     — tận dụng spatial structure
2. 🥈 CNNTransformerHybrid (~52%) — CNN + global attention
3. 🥉 ChannelTokenViT (~45%) — channel attention
4. SimpleViT/CustomViT (~40%)
5. LSTM/GRU variants (~38-42%)
6. SpatialTokenViT (~35%)  — bottlenecked bởi chất lượng pixel features
7. MLP (~37%)
8. SoftmaxRegression (~17%) — tuyến tính, underfitting